# Música em Contexto: Music Brainz + Spotify

## Camada Staging
Organização do ambiente, carga de dados brutos, análise inicial e primeiras decisões do projeto.

Organização do ambiente em camadas:

In [0]:
%skip
%sql
CREATE CATALOG mvp;

In [0]:
%skip
%sql
USE CATALOG mvp;
CREATE SCHEMA staging;
CREATE SCHEMA bronze;
CREATE SCHEMA silver;
CREATE SCHEMA gold;

##Staging

In [0]:
%skip
%sql
USE CATALOG mvp;
USE SCHEMA staging;
CREATE VOLUME imdb;

###Canonical Dump Music Brainz

Baixando e extraindo a base de dados Canonical Dump de Music Brainz.
Eu fiz o upload da base de dados musicbrainz-canonical-dump do meu computador para o imdb e agora estou extraindo o arquivo compactado para o mesmo diretório.

[https://musicbrainz.org/doc/Canonical_MusicBrainz_data](https://musicbrainz.org/doc/Canonical_MusicBrainz_data)

**INFORMAÇÕES DE LICENÇA**

MusicBrainz Canonical Data — MusicBrainz, MetaBrainz Foundation. Licença: Creative Commons Zero (CC0 1.0). Fonte: MusicBrainz Canonical MusicBrainz Data.

EXPLICAR AQUI MELHOR PORQUE EU RESOLVI COMEÇAR O ESTUDO COM ESSA BASE DE DADOS

In [0]:
%skip
%sh
cd /Volumes/mvp/staging/imdb
tar --use-compress-program=zstd -xvf musicbrainz-canonical-dump-20260903-080002.tar.zst

Mudando as tabelas e os arquivos de metadata de MusicBrainz para o diretório imdb:

In [0]:
%skip
%sh
cd /Volumes/mvp/staging/imdb

# Move CSV files from canonical subfolder to imdb directory
echo "Moving CSV files to imdb directory..."
mv musicbrainz-canonical-dump-20260903-080002/canonical/*.csv .

echo -e "\nCSV files now in imdb directory:"
ls -lh *.csv

In [0]:
%skip
%sh
cd /Volumes/mvp/staging/imdb

mv musicbrainz-canonical-dump-20260903-080002/COPYING .
mv musicbrainz-canonical-dump-20260903-080002/TIMESTAMP .

Removendo a antiga pasta que continha as tabelas de MusicBrainz:

In [0]:
%skip
%sh
cd /Volumes/mvp/staging/imdb

rm -rf musicbrainz-canonical-dump-20260903-080002

###Spotify Huge Audio Features

Aqui eu baixo a base de dados Spotify Huge Audio Features:

https://huggingface.co/datasets/GildasLeDrogoff/spotify-huge-track-analysis-dataset

**INFORMAÇÕES DE LICENÇA**

Spotify Huge Track Analysis Dataset, de Gildas Le Drogoff, disponibilizado sob a licença Creative Commons Attribution-NonCommercial 4.0 International (CC BY-NC 4.0).


EXPLICAR AQUI PORQUE EU RESOLVI INCLUIR ESSA BASE DE DADOS NO ESTUDO, O QUE ELA TROUXE PARA O ESTUDO?

In [0]:
%skip
dbutils.fs.cp(
    "https://huggingface.co/datasets/GildasLeDrogoff/spotify-huge-track-analysis-dataset/resolve/main/spotify-huge-audio-features.parquet",
    "/Volumes/mvp/staging/imdb/spotify-huge-audio-features.parquet"
)

### Music Brainz Area

Para ter acesso às informações de localização das músicas eu tive que buscar entre os **MusicBrainz JSON Data Dumps** https://musicbrainz.org/doc/Development/JSON_Data_Dumps os arquivos referentes à **Area**.

Eu baixei os arquivos para o meu computador, importei e extraí em staging/imdb

In [0]:
%skip
%sh
cd /Volumes/mvp/staging/imdb
tar --use-compress-program=zstd -xvf area.tar.xz

In [0]:
%skip
%sh
cd /Volumes/mvp/staging/imdb

rm -rf area.tar.xz

Movendo **area** para stagging/imdb e deletando a antiga pasta mbdump/ que a continha.

In [0]:
%skip
%sh
cd /Volumes/mvp/staging/imdb

echo "Movendo area para imdb"
mv mbdump/area .

echo -e "area está agora em imdb"
ls -lh area

In [0]:
%skip
%sh

cd /Volumes/mvp/staging/imdb
rm -rf mbdump/

### Music Brainz Artist

Após estudar a data frame **Area** ainda na etapa exploratória em **Staging** eu percebi que precisaria de informações complementares QUAIS INFORMALÇIES COMPLEMENTARES?????? para fazer o join com a base de dados principal, a **Canonical Music Brainz**, então baixei também a data frame **Artist** do pacote **Music Brainz JSON Data DUmps**  https://musicbrainz.org/doc/Development/JSON_Data_Dumps

**INFORMAÇÕES DE LICENÇA**

MusicBrainz Artist Data — MusicBrainz, MetaBrainz Foundation. Licença: Creative Commons Zero (CC0 1.0), para os dados core utilizados neste projeto.

In [0]:
%skip
%sh
cd /Volumes/mvp/staging/imdb
tar --use-compress-program=zstd -xvf artist.tar.xz

In [0]:
%skip
%sh
cd /Volumes/mvp/staging/imdb

# Move CSV files from canonical subfolder to imdb directory
echo "Movendo artist para imdb"
mv mbdump/artist .

echo -e "area está agora em imdb"
ls -lh artist

In [0]:
%skip
%sh

cd /Volumes/mvp/staging/imdb
rm -rf mbdump/
rm -rf artist.tar.xz

###Importando dependências##

Tive que instalar o pacote unidecode para conseguir converter nomes de algumas músicas e artistas na data frame Spotify da mesma forma que a Music Brainz fez para criar combined_lookup nas duas tabelas.

In [0]:
%pip install unidecode
%restart_python

Tive que importar UDF para conseguir usar a função python com unidecode que o Music Brainz usa para produzir o combined_lookup

In [0]:
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

In [0]:
import numpy as np

In [0]:
from pyspark.sql import functions as F

In [0]:
from unidecode import unidecode
import re

In [0]:
SEED = 42

### Nomeando e conhecendo as data frames

#### Music Brainz: Canonical Dump

Criando **df_musicbrainz** com a tabela Canonical Metadata e visualizando-a:__

In [0]:
df_musicbrainz = spark.read.csv(
    "/Volumes/mvp/staging/imdb/canonical_musicbrainz_data.csv",
    header=True,
    inferSchema=True
)

#display(df_musicbrainz.limit(10))
display(
    df_musicbrainz
    .sample(withReplacement=False, fraction=0.03, seed=SEED)
    .limit(200)
)

**combined_lookup**

Explicar aqui pq o combined lookup é importante para a pesquisa.

Infelizmente **exsitem algumas músicas nas quais os espaços e caixa alta foram mantidos**. Parecem ser todos nomes orientais pela amostra. Parece que foram mantidos os espaços dos nomes que foram traduzidos de alguma(s) lingua(s) orientais.

Preciso investigar as linhas que tem combined_lookup com espaços. Pegar uma amostra randomica de linhas com espaços. Contar quantas combined_lookup do total tem espaços e caixas altas.

Provavelmente o melhor que eu posso fazer é deletar esses espaços criando uma nova coluna na camada **Silver**

Criando **df_musicbrainz_release** com a Canonical Release Mapping e visualizando-a:

In [0]:
df_musicbrainz_release = spark.read.csv(
    "/Volumes/mvp/staging/imdb/canonical_release_redirect.csv",
    header=True,
    inferSchema=True
)

display(df_musicbrainz_release.limit(10))

Criando **df_musicbrainz_recording** com a Canonical Recording Mapping e visualizando-a:

In [0]:
df_musicbrainz_recording = spark.read.csv(
    "/Volumes/mvp/staging/imdb/canonical_recording_redirect.csv",
    header=True,
    inferSchema=True
)

display(df_musicbrainz_recording.limit(10))

#### MusicBrainz: Area JSON

Conhecendo o arquivo **area**

In [0]:

display(dbutils.fs.head("/Volumes/mvp/staging/imdb/area", 5000))

In [0]:
df_area = spark.read.json("/Volumes/mvp/staging/imdb/area")

display(df_area)

O que existe de interessante na **df_area**:

- id
- name
- short-name
- type
- type-id

In [0]:
area_shape = (df_area.count(), len(df_area.columns))

print(area_shape)

#### Music Brainz: Artist JSON

Agora o mesmo que eu fiz com **area** eu vou fazer com **artist**

In [0]:
df_artist = spark.read.json("/Volumes/mvp/staging/imdb/artist")

display(df_artist.limit(200))

O que existe de interessante na **df_artists**:

- area
- country
- gender
- gender-id
- id
- name
- sort-name
- type
- type-id

Dá para enxugar mais, claro. De repente é melhor simplificar.

**Investigar se usar apenas o país dessa tabela já resolve o problema.**

In [0]:
artist_shape = (df_artist.count(), len(df_artist.columns))

print(artist_shape)

#### Spotify Huge Audio Features

Criando df_spotify com Spotify Huge Audio Features e visualizando as dez primeiras linhas:

In [0]:
df_spotify = spark.read.parquet(
    "/Volumes/mvp/staging/imdb/spotify-huge-audio-features.parquet"
)

display(df_spotify.limit(10))

## Exploração e Avaliação Estratégica da Integração

### Testando a complexidade de conseguir correspondências

Através dos testes a seguir eu procuro descobrir qual a complexidade de conseguir correspondências entre as tabelas do Spotify e Canonical MusicBrainz Data.

Na primeira versão eu segui a forma como o GPT me sugeriu, tentando matches entre artistas, usando a data frame do Spotify como ponto de partida, o que foi um grande erro. Eu achei que a canonical funcionava em torno do nome padrão do artista, mas é, como faz muito mais sentido para a minha pesquisa, através de grupos de releases:

Recording = gravação específica

Release = um lançamento específico dessa música ou conjunto de músicas DÚVIDA NÃO ENTENDI DIRETO O QUE É RELEASEEE

Release Group = Agrupa releases qe representam a mesma música. O Canonical Release Mapping escolhe uma release representativa para cada release group.

combined_lookup = foi criado justamente para facilitar matching com bases de dados externas - então claramente vai ser meu ponto de partida, embora o GPT ainda não veja dessa forma.

Vou tentar matches com o Spotify usando como partida combined_lookup. Eu acredito que a canonical mb tem que ser meu ponto de partida de organização pq ela já está arquitetada para isso, não faz sentido usar o Spotify como ponto de partida.

Ao lidar com df_musicbrainz:

referência: https://musicbrainz.org/doc/Canonical_MusicBrainz_data

combined_lookup = artist_credit_name + recording_name -> sem pontuação e com todos os caracteres não ASCII convertidos para ASCII de acordo com https://pypi.org/project/Unidecode/

score = ordem de prioridade dos itens. Alguns sistemas de busca (como typesense) requerem que os dados estejam ordenados dessa forma https://typesense.org/

In [0]:
%sql
USE CATALOG mvp;
USE SCHEMA staging;

Primeiro eu quero saber quantas combined_lookup tem espaços e letras maiúsculas.

In [0]:
musicbrainz_shape = (df_musicbrainz.count(), len(df_musicbrainz.columns))

print(musicbrainz_shape)

In [0]:
contagem_esp_maiusc = (
    df_musicbrainz
    .select(
        F.count("*").alias("total"),
        F.sum(
            F.when(F.col("combined_lookup").contains(" "), 1).otherwise(0)
        ).alias("com_espacos"),
        F.sum(
            F.when(F.col("combined_lookup") != F.lower(F.col("combined_lookup")), 1).otherwise(0)
        ).alias("com_maiusculas"),
        F.sum(
            F.when(F.col("combined_lookup").contains(" ") |
                   (F.col("combined_lookup") != F.lower(F.col("combined_lookup"))), 1).otherwise(0)
                   ).alias("espacos_eou_maiusculas")
        )
        )
contagem_esp_maiusc = contagem_esp_maiusc.withColumn("porcentagem_do_total", F.lit(
    F.round(F.col("espacos_eou_maiusculas")*100/F.col("total"), 2)
    ))

display(contagem_esp_maiusc)

Então 5% de combined_lookup está com espaço e / ou letra maiúscula. O problema é que eu acredito que esses 5% representam uma porcentagem especialmente alta de músicas de origem oriental. Simplesmente excluir essas linhas poderia distorcer o resultado da pesquisa.

Então eu resolvi entender quais características das linhas que estão associadas a um combined_lookup não normalizado:

In [0]:
%skip
investigacao_anomalia_cl = (
    df_musicbrainz
    .select("combined_lookup")
    .where(F.col("combined_lookup").isNotNull())
    .distinct()
    .withColumn(
        "tem_espaco",
        F.col("combined_lookup").contains(" ")
    )
    .withColumn(
        "tem_maiuscula",
        F.col("combined_lookup") != F.lower(F.col("combined_lookup"))
    )
    .filter(
        F.col("tem_espaco") | F.col("tem_maiuscula")
    )
)

display(investigacao_anomalia_cl)

Vou pegar uma amostragem aleatória dessa tabela "investigacao_anomalia_cl" para ver se a maioria é oriental, mesmo.

In [0]:
%skip
display(
    investigacao_anomalia_cl
    .sample(withReplacement=False, fraction=0.03, seed=SEED)
    .limit(200)
)

Sim, a maioria parece ser oriental. Alguns também contém outros caracteres como ífens e aspas e acentuação.

A seguir verifico se existem muitos valores repetidos para "combined_lookup"

In [0]:
contagem_repetidos_cl = (
    df_musicbrainz
    .select(
        F.count("*").alias("total"),
        )
        )

contagem_repetidos_cl = contagem_repetidos_cl.withColumn("distintos", F.lit(df_musicbrainz.select("combined_lookup").distinct().count()))

contagem_repetidos_cl = contagem_repetidos_cl.withColumn("repetidos", F.lit(F.col("total")-F.col("distintos")))

contagem_repetidos_cl = contagem_repetidos_cl.withColumn("porcentagem_repetidos", F.lit(F.col("repetidos")*100 / F.col("total")))


display(contagem_repetidos_cl)

Então mesmo nessa coluna que ainda não está totalmente normalizada tenho uma quantidade de valores iguais... Será que vai aumentar muito quando normalizar tudo?

Vou ter que criar uma "combined_lookup" mais normalizada na própria df_musicbrainz "cl_normal"

Acabei criando uma outra df, df_musicbrainz_norm


Nesse primeiro teste eu criei uma outra coluna **cl_normal** usando como base o **combined_lookup** existente: tirei espaços e caixas altas que tinham resistido à normalização da Music Brainz.

Esse resultado serviu apenas como teste, depois eu percebi que fazia mais sentido criar a minha própria substituta de **combined_lookup** com regras que eu poderia aplicar idênticas nas informações de Music Brainz e do Spotify.

In [0]:
%skip
df_musicbrainz_norm = df_musicbrainz.withColumn(
    "cl_normal",
    F.lower(
        F.regexp_replace(F.col("combined_lookup"), r"\s+", "")
    )
)

display(
    df_musicbrainz_norm
    .sample(withReplacement=False, fraction=0.03, seed=SEED)
    .limit(200))

Criando uma função para aplicar idêntica na df_spotify:

In [0]:
def faz_normal(df, coluna_original, coluna_normalizada):
    return df.withColumn(
        coluna_normalizada,
        F.lower(
            F.regexp_replace(
                F.col(coluna_original),
                r"\s+",
                ""
            )
        )
    )

Acho que essa forma de normalização não tirou acentos nem caracteres especiais. Vale investigar isso pq eu vou ter que fazer a normalização do Spotify seguindo o mesmo padrão, ou então não vou conseguir correspondências.

Primeiro vendo se aumentou muito a quantidade de valores duplicados de combined_lookup para cl_normal:

In [0]:
contagem_repetidos_cl_normal = (
    df_musicbrainz_norm
    .select(
        F.count("*").alias("total"),
        )
        )

contagem_repetidos_cl_normal = contagem_repetidos_cl_normal.withColumn("distintos", F.lit(df_musicbrainz_norm.select("cl_normal").distinct().count()))

contagem_repetidos_cl_normal = contagem_repetidos_cl_normal.withColumn("repetidos_norm", F.lit(F.col("total")-F.col("distintos")))

contagem_repetidos_cl_normal = contagem_repetidos_cl_normal.withColumn("porcentagem_repetidos", F.lit(F.col("repetidos_norm")*100 / F.col("total")))


display(contagem_repetidos_cl_normal)
display(contagem_repetidos_cl)

Aumentou muito pouco.

EXPLICAR ESSA PARTE DIREITO. É QUANDO EU PERCEBO QUE É MELHOR EU FAZER MINHA PRÓPRIA NORMALIZAÇÃO CRIANDO O MEU PRÓPRIO COMBINED_LOOKUP

Depois ver SE e COMO combined_lookup tirou os cacarteres especiais das informações anteriores.

Para isso eu tenho que comparar os valores das duas colunas que foram unidas para formar o combined lookup e ver se muitas tinham caracteres especiais que foram tirados depois. De repente valia a pena saber quais caracteres são esses, sendo que na página que explica o canonical tem falando sobre isso...

referência: https://musicbrainz.org/doc/Canonical_MusicBrainz_data

combined_lookup = artist_credit_name + recording_name -> sem pontuação e com todos os caracteres não ASCII convertidos para ASCII de acordo com https://pypi.org/project/Unidecode/

"sem pontuação e com todos os caracteres não ASCII transformados em ASCII" - essa transformação eu vou ter que levar a cabo até o fim e aplicar exatamente igual na base de dados Spotify.

Então acho que eu vou ter que fazer mais uma coluna......... com um dummy combined_lookup apenas juntando os dois atributos e tirando espaços e letras maiúsculas, aí eu vou comparar esse dummy com o cl_normal e ver o quanto tem de diferente e o quê.

Então primeiro a criação do dummy_cl

In [0]:
%skip
df_musicbrainz_norm = df_musicbrainz_norm.withColumn(
    "dummy_cl",
    F.lower(
        F.concat(
            F.regexp_replace(F.col("artist_credit_name"), r"\s+", ""),
            F.regexp_replace(F.col("recording_name"), r"\s+", "")
        )
    )
)

display(
    df_musicbrainz_norm
    .sample(withReplacement=False, fraction=0.03, seed=SEED)
    .limit(200))

Ok, então vou continuar normalizando até o dummy_cl ficar igual ao cl_normal, e então vou criar a função para aplicar no Spotify.

Próxima etapa é converter os non-ASCII? Acho que vai ficar com aquele mesmo problema de vir com espaços. Acho que eu tenho que normalizar os espaçoes depois de normalizar os não-ASCII... e depois tirar os cacarteres especiais: - () . : & ? ! tbm tem um travessão que eu não sei como faze com o teclado aqui... será que tem como mapear todos esses caracteres que precisam ser retirados??...

Essa é a função exata que o Music Brainz usou para converter os nomes:

from unidecode import unidecode
import re

def make_combined_lookup(artist_name, recording_name):
    return unidecode(
        re.sub(r'[^\w]+', '', artist_name + recording_name).lower()
    )

Então acho que o ideal agora é apagar essa coluna dummy_cl que eu criei e criar uma noma com esse cóodigo para ver como fica. Tem que ficar igual a primeira versão de combined_lookup, sem a última normalização.

In [0]:
%skip
df_musicbrainz_norm = df_musicbrainz_norm.drop("dummy_cl")

display(df_musicbrainz_norm)

Toda vez que aplicar uma nova normalização preciso ver se o número de valores repetidos aumentou demais....

Usando a função que o music brainz usou para criar combined_lookup, tenho que fazer agora uma nova dummy_cl que trace o processo que preciso aplicar no Spotify.

Então primeiro vou testar essa função para uma dummy_cl e essa dummy tem que ficar igual a combined_lookup antes da normalização que eu fiz depois.

In [0]:
def faz_cl(artist_name, recording_name):
    return unidecode(
        re.sub(
            r'[^\w]+',
            '',
            artist_name + recording_name
        ).lower()
    )

In [0]:
%skip
print(faz_cl("Beyoncé", "Déjà Vu"))
print(faz_cl("東京", "音楽"))

sim, está acontecendo a mesma coisa. As palavras que são traduzidas estão saindo com espaços e letras maiúsculas e vai precisar passar aquela segunda normalização de novo. Mas isso é um bom sinal.

Então agora vou refazer a dummy na ordem certa e vai ficar igual, acredito.

Só que a função python não consegue lidar com as colunas direito... ela fica procurando uma sting e recebe uma coluna. Precisa usar UDF, que eu coloquei nas importações lá em cima, para ficar organizado.

In [0]:
faz_cl_udf = udf(
    faz_cl,
    StringType()
)

Agora sim criando a dummy_cl com a função faz_cl_udf:

In [0]:
df_musicbrainz_norm = df_musicbrainz_norm.withColumn(
    "dummy_cl",
    faz_cl_udf(
        F.col("artist_credit_name"),
        F.col("recording_name")
    )
)

display(
    df_musicbrainz_norm
    .sample(withReplacement=False, fraction=0.03, seed=SEED)
    .limit(200))

Agora vou conferir se essa versão de dummy_cl está igual a primeira versão de combined_lookup:

In [0]:
dummy1_combined = df_musicbrainz_norm.filter(
    F.col("combined_lookup") != F.col("dummy_cl")
).count()

total = df_musicbrainz_norm.count()

print(f"São {dummy1_combined} diferentes em {total}, ou seja, {dummy1_combined/total*100}%")

Diferença muito pequena entre o meu **dummy_cl** e a versão original de **combined_lookup**. Essa normalização vai ser usada para construir um **combined_lookup** similar na **df_spotify**, através do qual vamos procurar as correspondências e fazer o join.

Aqui eu produzo uma amostra randomica de 200 linhas nas quais o **dummy_cl** produzido por mim não ficou idêntico ao **combined_lookup** fornecido pela Music Brainz, para investigar o que pode estar acontecendo:

In [0]:
in_dummy1_combined = df_musicbrainz_norm.filter(
    F.col("combined_lookup") != F.col("dummy_cl")
)

display(
    in_dummy1_combined
    .sample(withReplacement=False, fraction=0.03, seed=SEED)
    .limit(200)
    .drop("cl_normal", "score", "artist_credit_id", "artist_mbids", "release_name", "release_mbid", "recording_mbid"))

Observando essa amostragem acho que está fazendo bem mais sentido o meu dummy_cl. De repente produzir o meu próprio cl vai funcionar melhor. Mas posso fazer e testar com os dois... o match com o Spotify.

Acho que eu vou fazer um novo cl_normal com dummy_cl + aquela outra função que tira espaços e maiusculas de novo e vou comparar essa e combined_lookup com o cl_normal que eu obtiver no Spotify para ver com qual eu consigo mais match.

In [0]:
df_musicbrainz_norm = df_musicbrainz_norm.drop("cl_normal")

display(
    df_musicbrainz_norm
    .sample(withReplacement=False, fraction=0.03, seed=SEED)
    .limit(200)
)

Ok, agora vou criar a nova cl_norm passando a segunda função de normalização na dummy_cl.

Aí vou usar essas duas funções para normalizar as infos do Spotify e tentar uma correspondência.... Será que vai dar mais do que 30%?.......

Funções:

faz_cl_udf(
        F.col("artist_credit_name"),
        F.col("recording_name")
    )


faz_normal(df, coluna_original, coluna_normalizada):
    return df.withColumn(
        coluna_normalizada,
        F.lower(
            F.regexp_replace(
                F.col(coluna_original),
                r"\s+",
                ""
            )
        )
    )


In [0]:
df_musicbrainz_norm = faz_normal(df_musicbrainz_norm, "dummy_cl", "cl_normal")

display(
    df_musicbrainz_norm
    .sample(withReplacement=False, fraction=0.03, seed=SEED)
    .limit(200)
    .drop("artist_credit_id", "artist_mbids", "release_name", "release_mbid", "recording_mbid", "score")
    )

Posso aproveitar aqui e fazer aquele teste para ver se existem muitos valores repetidos em cl_normal

In [0]:
contagem_repetidos_cl_normal = (
    df_musicbrainz_norm
    .select(
        F.count("*").alias("total"),
        )
        )

contagem_repetidos_cl_normal = contagem_repetidos_cl_normal.withColumn("distintos", F.lit(df_musicbrainz_norm.select("cl_normal").distinct().count()))

contagem_repetidos_cl_normal = contagem_repetidos_cl_normal.withColumn("repetidos_norm", F.lit(F.col("total")-F.col("distintos")))

contagem_repetidos_cl_normal = contagem_repetidos_cl_normal.withColumn("porcentagem_repetidos", F.lit(F.col("repetidos_norm")*100 / F.col("total")))


display(contagem_repetidos_cl_normal)
display(contagem_repetidos_cl)

Lá em cima era 0,189%. Da forma como eu fiz diminuiu a porcentagem para 0,156%

Muito bom

Então cl_normal é o padrão ouro dessa base de dados que eu vou usar para comunicar com a base de dados Spotify.

Ok, então agora eu tenho que fazer o mesmo processo para o Spotify, criando uma spotfy_dummy_cl e uma spotfy_cl_normal.

In [0]:
display(df_spotify.limit(10))

Criando uma df_spotify_norm com um spotify_dummy_cl criado com a função faz_cl_udf

In [0]:
df_spotify_norm = df_spotify.withColumn(
    "spotify_dummy_cl",
    faz_cl_udf(
        F.col("artist_name"),
        F.col("track_name")
    )
)

display(
    df_spotify_norm
    .sample(withReplacement=False, fraction=0.03, seed=SEED)
    .limit(200))

Investigando a quantiade de nomes repetidos em spotify_dummy_cl:

In [0]:
contagem_repetidos_sdcl = (
    df_spotify_norm
    .select(
        F.count("*").alias("total"),
        )
        )

contagem_repetidos_sdcl = contagem_repetidos_sdcl.withColumn("distintos", F.lit(df_spotify_norm.select("spotify_dummy_cl").distinct().count()))

contagem_repetidos_sdcl = contagem_repetidos_sdcl.withColumn("repetidos_norm", F.lit(F.col("total")-F.col("distintos")))

contagem_repetidos_sdcl = contagem_repetidos_sdcl.withColumn("porcentagem_repetidos", F.lit(F.col("repetidos_norm")*100 / F.col("total")))


display(contagem_repetidos_sdcl)

A porcentagem de repetidos é bastante grande comparada com a Music Brainz, com certeza pelo fato a Musica Brainz ser canonical.

Vpu fazer a segunda parte da normalização e medir os repetidos de novo. Depois decido o que vou fazer com esses repetidos.

Usando faz_normal para transformar spotify_dummy_cl em spotify_cl_normal em df_spotify_norm

In [0]:
df_spotify_norm = faz_normal(df_spotify_norm, "spotify_dummy_cl", "spotify_cl_normal")

display(
    df_spotify_norm
    .sample(withReplacement=False, fraction=0.03, seed=SEED)
    .limit(200)
    .drop()
    )

Ok, agora eu vou comparar os retepitos de spotify_cl_normal com os repetidos em spotify_dummy_cl

In [0]:
contagem_repetidos_sdcl_norm = (
    df_spotify_norm
    .select(
        F.count("*").alias("total"),
        )
        )

contagem_repetidos_sdcl_norm = contagem_repetidos_sdcl_norm.withColumn("distintos", F.lit(df_spotify_norm.select("spotify_cl_normal").distinct().count()))

contagem_repetidos_sdcl_norm = contagem_repetidos_sdcl_norm.withColumn("repetidos_norm", F.lit(F.col("total")-F.col("distintos")))

contagem_repetidos_sdcl_norm = contagem_repetidos_sdcl_norm.withColumn("porcentagem_repetidos", F.lit(F.col("repetidos_norm")*100 / F.col("total")))


display(contagem_repetidos_sdcl_norm)
display(contagem_repetidos_sdcl)

Que bom, praticamente a mesma coisa. Agora só tenho que ver o que são esses repetidos e o que fazer com eles.

Vale a pena ver qual a diferença do número de linhas para cada base de dados.

In [0]:
spotify_norm_shape = (df_spotify_norm.count(), len(df_spotify_norm.columns))


print(f"shape musicbrainz: {musicbrainz_shape}")
print(f"shape spotify: {spotify_norm_shape}")

#Agora calcula a diferença de tamanho em %

print(f"diferença shape: {((spotify_norm_shape[0] - musicbrainz_shape[0]) / spotify_norm_shape[0]) * 100}%")

Ok, a spotify é muito maior. Vamos ver qual o percentual de conrrespondências possível agora. Fazer o percentual comparando o a base de dados menor, claro.

### Gravando df em Bronze e Silver

Aqui eu já tenho mais clareza de quais df eu vou realmente usar nesse estudo e vou passá-las para a camada **Bronze**.

Também já tenho duas df modificadas que vou limpar e passar para a camada **Silver** para continuar modificando.

Dos dados que eu baixei e investiguei na camada **Staging**, vou passar para a camada **Bronze** apenas aqueles que realmente serão usados como base do estudo:

- a df_musicbrainz
- a df_artist
- a df_spotify

E apenas isso será passado, sem modificação, para a camada bronze.

**PAREI AQUI**

In [0]:
%skip

# EDITAR ESSA CELULA COM AS DF QUE EU VOU PASSAR PARA BRONZE!!!!

df_spotify.write.mode("overwrite").saveAsTable(
    "mvp.bronze.spotify"
)

df_canonical.write.mode("overwrite").saveAsTable(
    "mvp.bronze.musicbrainz_canonical"
)

df_artists.write.mode("overwrite").saveAsTable(
    "mvp.bronze.musicbrainz_artist"
)

### Procurando correspondências entre as duas bases de dados:

Primeiro vou criar uma amostra pequena da coluna cl_norm da base de dados df_musicbrainz_norm

Depois vou procurar esses nomes na coluna spotify_cl_norm na df_spotify_norm

Medir quantas correspondências eu consegui e qual porcentagem elas representam do total na mini amostra

In [0]:
mini_mb_norm100 = (
    df_musicbrainz_norm
    .select("cl_normal", "recording_mbid")
    .where("cl_normal IS NOT NULL")
    .distinct()
    .sample(withReplacement=False, fraction=0.001, seed=SEED)
    .limit(100)
)

display(mini_mb_norm100)

In [0]:
mini_exact_matches = (
    mini_mb_norm100
    .join(
        df_spotify_norm,
        mini_mb_norm100.cl_normal == df_spotify_norm.spotify_cl_normal,
        how="inner"
    )
)

display(mini_exact_matches)

In [0]:
%skip
mini_exact_matches = exact_matches

del exact_matches

Ok, eu acho que eu vou fazer uma tabela assim com todas as correspondências que eu conseguir e depois vou comparar essa tabela com a music brainz original.

In [0]:
df_mb_spotify_1 = (
    df_musicbrainz_norm
    .join(
        df_spotify_norm,
        df_musicbrainz_norm.cl_normal == df_spotify_norm.spotify_cl_normal,
        how="inner"
    )
)

display(
    df_mb_spotify_1
    .sample(withReplacement=False, fraction=0.03, seed=SEED)
    .limit(200)
    .drop()
    )

Não estou entendendo pq a df_artist display só está me mostrando sempre duas linhas, apenas.... meio assustador. Ver o que é isso.

Vou jogar fora algumas colunas que eu sei que não vou precisar. E depois vou comparar o tamanho dessa df com o tamanho da df musicbrainz

Então num primeiro momento eu estou jogando fora as colunas que eu criei para normalizar os nomes e a coluna score.

In [0]:
%skip
df_mb_spotify_1 = df_mb_spotify_1.drop("combined_lookup", "score", "dummy_cl", "spotify_dummy_cl")

Agora vou comparar o tamanho da df_mb_spotify_1 com df_musicbrainz_norm

In [0]:
mb_spotify_1_shape = (df_mb_spotify_1.count(), len(df_mb_spotify_1.columns))


print(f"shape mb_spotify_1: {mb_spotify_1_shape}")
print(f"shape musicbrainz: {musicbrainz_shape}")
print(f"shape spotify: {spotify_norm_shape}")

print(f"diferença shape: {(musicbrainz_shape[0] - mb_spotify_1_shape[0])}")
print(f"diferença shape %: {((musicbrainz_shape[0] - mb_spotify_1_shape[0]) / musicbrainz_shape[0]) * 100}%")
print(f"porcentagem que conseguimos correspondência: {((mb_spotify_1_shape[0]) / musicbrainz_shape[0]) * 100}%")

**PAREI AQUI**

**TALVEZ USAR APENAS O PAÌS DA DF_ARTIST JÁ FUNCIONE PARA O QUE EU QUERO**

Assim eu não precisaria ter que lidar com toda a df_area

In [0]:
%skip
display(df_area.limit(100))

In [0]:
display(df_artist.limit(100))

Ok, vou fazer uma tabela de artistas só com as informações que eu acho que vou precisar.

In [0]:
df_artist_trim = df_artist.select(
    "id",
    "name",
    "gender",
    "country"
)

In [0]:
display(df_artist_trim.limit(10))

In [0]:
df_artist_trim.groupBy("id").count().filter(F.col("count") > 1).count()

In [0]:
country_null = df_artist_trim.filter(
    F.col("country").isNull()
).count()

print(country_null)
print(country_null / df_artist_trim.count() * 100)

In [0]:
area_null = df_artist.filter(
    F.col("area").isNull()
).count()

print(area_null)
print(area_null / df_artist.count() * 100)

Existe uma porcentagem grande de valores nulos para "country" e para "area"

Agora vou unir a df_mb_spotify_1 com as informações de artist_trim

In [0]:
df_mb_spotify_2 = (
    df_mb_spotify_1
    .join(
        df_artist_trim,
        df_mb_spotify_1["artist_mbids"] == df_artist_trim["id"],
        "left"
    )
    .filter(F.col("country").isNotNull())
)

In [0]:
display(df_mb_spotify_2.limit(100))

Agora vou ver quantas linhas eu perdi comparando com a df_mb_spotify_1

In [0]:
mb_spotify_2_shape = (df_mb_spotify_2.count(), len(df_mb_spotify_2.columns))


print(f"shape mb_spotify_1: {mb_spotify_1_shape}")
print(f"shape mb_spotify_2: {mb_spotify_2_shape}")

print(f"shape musicbrainz: {musicbrainz_shape}")
print(f"shape spotify: {spotify_norm_shape}")

print("Comparando df_mb_spotify_1 e df_mb_spotify_2")
print(f"diferença shape: {(mb_spotify_1_shape[0] - mb_spotify_2_shape[0])}")
print(f"diferença shape %: {((mb_spotify_1_shape[0] - mb_spotify_2_shape[0]) / mb_spotify_1_shape[0]) * 100}%")
print(f"porcentagem que conseguimos correspondência: {((mb_spotify_2_shape[0]) / mb_spotify_1_shape[0]) * 100}%")

Consegui o país para 88% da base de dados df_mb_spotify_1
Só calcular aqui tudo que perdemos da MusicBrainz desde o início do processo:

In [0]:
print(f"spotify - mb = {56277664 - 31962546}")
print(f"mb - df final = {31962546 - 9370575}")
print(f"porcentagem que não foi aproveitada da mb: {(31962546 - 9370575)/31962546*100}")
print(f"porcentagem que não foi aproveitada de spotify: {(56277664 - 9370575)/56277664*100}")

Acabou que pouca coisa foi aproveitada... Será que isso é um problema?

"Após as etapas de filtragem e correspondência, 30% dos registros da base inicial chegaram à base analítica final."

Tá então o que provavelmente vai acontecer é o seguinte:

- eu vou formar uma nova df com as infos de artist COUNTRY
- ver as formas de fazer o join: inner, left, right, outer, left_semi, left_anti
- vou explorar essa df para ver como ela ficou: quais países eu tenho, qual período de tempo etc... FAZER GRAFICOS
- Depois vou achar uma base de dados que tenha índices economicos para esse período e países - SERá QUE VAI DAR
- e / ou só tentar fazer conexão com fatos históricos, guerras etc

- tenho que consultar o guis do MVP
- colocar na arquitetura medalhão: rever o vídeo da aula que estava me ajudando bastante
- fazer um notebook limpo para exportar para html
- linkar esse html no github e ativar o github pages

estrutura do relatório pelo GPT se eu for exportar em PDF

objetivo → dados → tratamento → integração Spotify/MusicBrainz → metodologia → resultados → limitações → conclusões.

**DEPOIS EU TENHO QUE COMPARAR ESSA TABELA COM TODAS AS CORRESPONDÊNCIAS E A TABELA MUSICBRAINZ PARA VER SE MUDA MUITO AS CARACTERÍSTICAS DAS MÚSICAS... TIPO SE DIMINUI MUITO MÚSICAS DE UM LOCAL, DADAS OU GÊNEROS ESPECÍFICOS A PONTO DE ACABAR INTERFERINDO NO RESULTADO DA PESQUISA** 

Infelizmente eu não vou ter como saber isso pq não vou colocar as infos de area na base de dados inteira... Vai ficar para a próxima...

Dar uma olhada de novo no MVP para ver se está faltando algo.

Construir uma única tabela com todas as informações que eu quero,

fazer alguma exploração inicial para entender qual período histórico e locais geográficos estou lidando.

Aí tentar achar uma base de dados com informações socioeconomicas e fatos históricios para cruzar e ver quais informações interessantes eu acho.

Depois de associar gêneros e características sonoras com acontecimentos históricos ou socio-economicos, tentar traçar o perfil dos dias de hoje.